# TRELLIS v1 — textured prop generator (16 GB+ GPU)

Generates ONE textured 3D prop from a single reference image, using **TRELLIS v1** (Microsoft, CVPR'25).

- **Licence:** MIT code + MIT weights (repo: `microsoft/TRELLIS`).
- **GPU — READ FIRST:** the official README requires **an NVIDIA GPU with at least 16 GB** (verified on A100/A6000). A free **Colab T4 (16 GB) sits exactly on the floor and risks OOM** — prefer **Kaggle P100 (16 GB)** or HF ZeroGPU; on a T4, use the `image-large` model only, one image, no batch. TRELLIS.2 needs 24 GB for PBR — out of scope here.
- **GLB export licence flag:** the GLB exporter uses **nvdiffrast**, which is research/evaluation-only — **do not ship TRELLIS GLBs commercially without replacing the exporter**. For this free project the outputs are prototypes; re-check before any commercial release.
- **Honest limits:** image-to-3D only (the README itself recommends generating an image first, then image-to-3D — text models are weaker). Output is a textured GLB but still needs Blender cleanup + decimation. **Props only, never the hero, never a face.**

Steps: 1) check GPU (warns below 16 GB), 2) clone + install (conda route from the official README — takes 20–40 min), 3) upload image + inference → `prop.glb`, 4) verify outputs.

In [ ]:
# 1) GPU check — TRELLIS needs 16 GB+ (A100/A6000 verified). T4 = borderline.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total_gb:.1f} GB')
    if total_gb < 16:
        print('WARNING: <16 GB VRAM — TRELLIS will likely OOM. Use Kaggle P100 (16 GB) instead of this runtime.')
    else:
        print('VRAM OK (>=16 GB)')

In [ ]:
# 2) Clone TRELLIS + install via the official setup.sh (conda route, 20-40 min)
# Colab has no conda by default: install Miniconda via condacolab first.
!pip install -q condacolab
import condacolab
condacolab.install()
print('conda ready - runtime will restart once, then re-run this notebook from cell 2')

In [ ]:
# 2b) Clone + run the official installer (run AFTER the condacolab restart)
%cd /content
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
%cd /content/TRELLIS
!bash setup.sh --new-env --basic --xformers --flash-attn --diffoctreerast --spconv --mipgaussian --kaolin --nvdiffrast
print('install done — the trellis conda env is ready')

In [ ]:
# 3) Inference: upload ONE reference image -> prop.glb (textured)
from google.colab import files
print('Upload one reference image (PNG/JPG, object on a plain background):')
up = files.upload()
img_path = '/content/' + list(up.keys())[0]
print('input:', img_path)

import os
os.environ['SPCONV_ALGO'] = 'native'  # recommended when running once (README minimal example)
%cd /content/TRELLIS
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

# NOTE: run this cell with the trellis conda env active (see setup.sh --new-env).
# In Colab, use: !conda run -n trellis python -c ... — or run this notebook inside the env.
pipeline = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipeline.cuda()
image = Image.open(img_path)
outputs = pipeline.run(image, seed=1)
glb = postprocessing_utils.to_glb(
    outputs['gaussian'][0],
    outputs['mesh'][0],
    simplify=0.95,          # ratio of triangles to remove
    texture_size=1024,      # GLB texture size
)
glb.export('/content/prop-trellis.glb')
print('exported /content/prop-trellis.glb')

In [ ]:
# 4) Verify outputs
import os
for f in ['/content/prop-trellis.glb']:
    print(f, os.path.getsize(f) if os.path.exists(f) else 'MISSING', 'bytes')
!ls -la /content/*.glb
print('Next: import the GLB in Blender, decimate to ~10k faces, hand-check the texture, then export to Godot.')